In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, count, avg, row_number
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark berhasil dijalankan!", spark.version)

26/10/14 10:50:02 WARN Utils: Your hostname, indra resolves to a loopback address: 127.0.1.1; using 192.168.1.19 instead (on interface wlo1)
26/10/14 10:50:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/14 10:50:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark berhasil dijalankan! 3.5.9


In [4]:
path_raw = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw"

# Membaca data transaksi
df_transaksi = spark.read.csv(
    f"{path_raw}/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data cabang
df_cabang = spark.read.json(
    f"{path_raw}/tugas7_cabang.json"
)

# Membaca data menu
df_menu = spark.read.csv(
    f"{path_raw}/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

print("Data berhasil di extract!")

Data berhasil di extract!


In [5]:
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

print("Jumlah data setelah join:", df_gabungan.count())
df_gabungan.show(10)

Jumlah data setelah join: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan

In [6]:
penjualan_menu = df_gabungan.groupBy(
    "cabang_id",
    "nama_cabang",
    "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy(
    "cabang_id"
).orderBy(
    col("total_penjualan").desc()
)

top2_menu = penjualan_menu.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
).orderBy(
    "nama_cabang",
    "peringkat"
)

top2_menu.show(truncate=False)

+---------+-------------------------+---------+---------------+---------+
|cabang_id|nama_cabang              |nama_menu|total_penjualan|peringkat|
+---------+-------------------------+---------+---------------+---------+
|1        |Kopi Nusantara Magelang  |Menu-1   |8505000        |1        |
|1        |Kopi Nusantara Magelang  |Menu-12  |8085000        |2        |
|3        |Kopi Nusantara Semarang  |Menu-13  |8750000        |1        |
|3        |Kopi Nusantara Semarang  |Menu-9   |8715000        |2        |
|2        |Kopi Nusantara Yogyakarta|Menu-12  |9275000        |1        |
|2        |Kopi Nusantara Yogyakarta|Menu-15  |8160000        |2        |
+---------+-------------------------+---------+---------------+---------+



In [7]:
df_gabungan.createOrReplaceTempView("penjualan_kopi")

hasil_sql = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM penjualan_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

hasil_sql.show(truncate=False)

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|Nadia         |177680000      |
|Reza          |84645000       |
+--------------+---------------+



In [9]:
path_output = (
    "hdfs://localhost:9000/user/mahasiswa/"
    "tugas7/processed/laporan"
)

df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(path_output)

print("Data berhasil disimpan dalam format Parquet.")

# Verifikasi
df_parquet = spark.read.parquet(path_output)

print("Jumlah baris sebelum disimpan:", df_gabungan.count())
print("Jumlah baris setelah dibaca:", df_parquet.count())

df_parquet.show(5)

Data berhasil disimpan dalam format Parquet.
Jumlah baris sebelum disimpan: 4000
Jumlah baris setelah dibaca: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12| 

In [10]:
ringkasan_eksekutif = df_gabungan.groupBy(
    "cabang_id",
    "nama_cabang",
    "kota",
    "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_eksekutif.show(truncate=False)

+---------+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|cabang_id|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|3        |Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|1        |Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|2        |Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+---------+-------------------------+----------+--------------+---------------+----------------+-------------------------+

